In [2]:
## 경고 제거##
import warnings
warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

## DataFrame & visual option ##
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_columns', None)
import polars as pl

import numpy as np
import os
import json

## geo ##
import geopandas as gpd

from tqdm import tqdm
## Visualize option ##
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.family'] = 'D2Coding'

# Preprocessing Intro
- 응시 Master에서 응시자 유형 분류 후 응시 ID 값에 따라서 데이터 결합
- 응시횟수가 2회 이상인 경우의 응시결과는 어떻게?
    > 응시횟수와 응시차수에 대한 컬럼을 만들자
- 필기시험에서는
- 기능시험에서 PNT는 감점 항목표시는 맞으나 트레일러, 렉카, 원동기 감점 항목도 존재하기 때문에 drop해서 사용
- 도로주행의 별도 점수는 지점별 감점 항목이기 때문에 문제별로 보기 위해서는 JOIN이 필요없을 것으로 보임

# Master
- 마스터가 필요할까?

# PC SUBJECT
- 우선 U_INFO에서 사람을 먼저 추린뒤에 JOIN TABLE에서 골라내기

In [25]:
subj = pd.read_csv('../../../011.데이터/채점데이터/_SELECT_ui_U_NAME_ui_U_JUMIN_NO_uec_U_INFO_ID_uec_Q_ID_qq_L_CD_q_20231101.csv',nrows=300000)
subj_mst = pd.read_csv('../../../011.데이터/채점데이터/U_INFO_202311011052.csv')

In [26]:
subj

,U_NAME,U_JUMIN_NO,U_INFO_ID,Q_ID,L_CD,M_CD,UNITCODE_NAME,Q_COUNT,CORR_ANSWER,U_ANSWER,CORRECT_YN,Q_LEVEL,U_EXAM_OPEN,U_TYPE,U_CUTLINE
0,홍정은,9309012******,35021,781,19,26,도로,28,"1,5","1,5",1,0,2022-01-03 09:00:00.000,2,60
1,홍정은,9309012******,35021,801,19,27,도로,29,"1,3","1,3",1,0,2022-01-03 09:00:00.000,2,60
2,홍정은,9309012******,35021,816,19,28,도로,30,"2,3","2,3",1,0,2022-01-03 09:00:00.000,2,60
3,홍정은,9309012******,35021,826,19,29,도로,31,"3,5","5,3",1,0,2022-01-03 09:00:00.000,2,60
4,홍정은,9309012******,35021,838,19,30,도로,32,"2,4","2,3",0,0,2022-01-03 09:00:00.000,2,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299995,차보경,0102074******,42917,825,19,29,도로,31,"1,5","5,3",0,0,2022-01-24 09:00:00.000,2,60
299996,차보경,0102074******,42917,837,19,30,도로,32,"1,5","5,1",1,0,2022-01-24 09:00:00.000,2,60
299997,차보경,0102074******,42917,844,20,31,인명,33,"2,4","4,2",1,0,2022-01-24 09:00:00.000,2,60
299998,차보경,0102074******,42917,854,17,21,"자전거, 친환경",34,"3,5","5,3",1,0,2022-01-24 09:00:00.000,2,60


In [214]:
subj = pd.read_csv('../../../011.데이터/채점데이터/_SELECT_ui_U_NAME_ui_U_JUMIN_NO_uec_U_INFO_ID_uec_Q_ID_qq_L_CD_q_20231101.csv',nrows=300000)
subj_mst = pd.read_csv('../../../011.데이터/채점데이터/U_INFO_202311011052.csv')

subj['BIRTH_NUM'] = subj['U_JUMIN_NO'].apply(lambda x: x[:6])
subj['PER_IDX'] = subj['U_NAME']+'_'+subj['BIRTH_NUM']

mask1= subj_mst['U_LEVEL'] == '보통'
mask2 = subj_mst['U_PASSFAIL'] == 1
subj_mst = subj_mst[mask1&mask2]
subj_mst = subj_mst[['ID','U_EXAM_OPEN','U_NAME', 'U_TYPE', 'U_LEVEL',]]

#사람별 정오답개수
subj = subj[subj['U_INFO_ID'].isin(subj_mst['ID'].unique())]
result_df = pd.DataFrame()
for idx in tqdm(subj['U_INFO_ID'].unique()):
    temp = subj[subj['U_INFO_ID'] == idx]
    vs = sorted(temp['UNITCODE_NAME'].unique())
    if vs != ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황']:
        continue    
    try:
        temp_pivot = temp.groupby(['U_INFO_ID','UNITCODE_NAME','CORRECT_YN'],as_index=False).agg({'U_NAME':'count'}).pivot(index='CORRECT_YN',columns='UNITCODE_NAME',values='U_NAME').fillna(0).reset_index()
        if temp_pivot.shape[0] > 1:
            temp_pivot.columns.name = None
            temp_pivot.reset_index(drop=True,inplace=True)
            temp_pivot.loc[2,:] = temp_pivot.sum().to_list()
            temp_pivot.iloc[2,0] = 'total'
            temp_pivot['U_INFO_ID'] = idx
            result_df = pd.concat([result_df,temp_pivot])
        elif temp_pivot.shape[0] == 1:
            temp_pivot.columns.name = None
            temp_pivot.reset_index(drop=True,inplace=True)
            temp_pivot['U_INFO_ID'] = idx
            temp_pivot.loc[1,:] = 0
            temp_pivot.sort_values('CORRECT_YN',ascending=True,inplace=True)
            temp_pivot.reset_index(drop=True,inplace=True)
            temp_pivot.loc[2,:] = temp_pivot.sum().to_list()
            temp_pivot.iloc[2,0] = 'total'
            temp_pivot['U_INFO_ID'] = idx
            result_df = pd.concat([result_df,temp_pivot])
            
            
    except IndexError:
        print(idx)
        pass

result_df['U_INFO_ID'] = result_df['U_INFO_ID'].astype('int')
result_df.reset_index(drop=True, inplace=True)


# 오답총합개수
fin_result = pd.DataFrame()
for uid in tqdm(result_df['U_INFO_ID'].unique()):
    new_temp = pd.DataFrame(columns = ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황'])
    new_temp.loc[0] = 0
    temp = result_df[result_df['U_INFO_ID'] == uid]

    for col in ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황']:
        new_temp[col] = round(temp[temp['CORRECT_YN'] == 0][col].iloc[0] / temp[temp['CORRECT_YN'] == 'total'][col].iloc[0], 2)
    new_temp['UID'] = uid
    fin_result = pd.concat([fin_result,new_temp])

fin_result = fin_result[['UID','교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명',
       '자전거, 친환경', '점검, 관리', '특별상황',]]

subj_result = pd.merge(subj_mst,fin_result, left_on = 'ID', right_on = 'UID' ,how='left')
#subj_result.to_csv('./subj_result.csv',index=False)

100%|███████████████████████████████████████████████████████████████████████████████| 6926/6926 [01:33<00:00, 74.19it/s]


In [9]:
## 경고 제거##
import warnings
warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

## DataFrame & visual option ##
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm


subj = pd.read_csv('../../../011.데이터/채점데이터/_SELECT_ui_U_NAME_ui_U_JUMIN_NO_uec_U_INFO_ID_uec_Q_ID_qq_L_CD_q_20231101.csv',nrows=300000)
subj_mst = pd.read_csv('../../../011.데이터/채점데이터/U_INFO_202311011052.csv')
print('End  -- read file --')

subj['BIRTH_NUM'] = subj['U_JUMIN_NO'].apply(lambda x: x[:6])
subj['PER_IDX'] = subj['U_NAME']+'_'+subj['BIRTH_NUM']

mask1= subj_mst['U_LEVEL'] == '보통'
mask2 = subj_mst['U_PASSFAIL'] == 1
subj_mst = subj_mst[mask1&mask2]
subj_mst = subj_mst[['ID','U_EXAM_OPEN','U_NAME', 'U_TYPE', 'U_LEVEL',]]

#사람별 정오답개수
subj = subj[subj['U_INFO_ID'].isin(subj_mst['ID'].unique())]
result_df = pd.DataFrame()
for idx in tqdm(subj['U_INFO_ID'].unique()):
    temp = subj[subj['U_INFO_ID'] == idx]
    vs = sorted(temp['UNITCODE_NAME'].unique())
    if vs != ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황']:
        continue    
    try:
        temp_pivot = temp.groupby(['U_INFO_ID','UNITCODE_NAME','CORRECT_YN'],as_index=False).agg({'U_NAME':'count'}).pivot(index='CORRECT_YN',columns='UNITCODE_NAME',values='U_NAME').fillna(0).reset_index()
        if temp_pivot.shape[0] > 1:
            temp_pivot.columns.name = None
            temp_pivot.reset_index(drop=True,inplace=True)
            temp_pivot.loc[2,:] = temp_pivot.sum().to_list()
            temp_pivot.iloc[2,0] = 'total'
            temp_pivot['U_INFO_ID'] = idx
            result_df = pd.concat([result_df,temp_pivot])
        elif temp_pivot.shape[0] == 1:
            temp_pivot.columns.name = None
            temp_pivot.reset_index(drop=True,inplace=True)
            temp_pivot['U_INFO_ID'] = idx
            temp_pivot.loc[1,:] = 0
            temp_pivot.sort_values('CORRECT_YN',ascending=True,inplace=True)
            temp_pivot.reset_index(drop=True,inplace=True)
            temp_pivot.loc[2,:] = temp_pivot.sum().to_list()
            temp_pivot.iloc[2,0] = 'total'
            temp_pivot['U_INFO_ID'] = idx
            result_df = pd.concat([result_df,temp_pivot])
            
            
    except IndexError:
        print(idx)
        pass

result_df['U_INFO_ID'] = result_df['U_INFO_ID'].astype('int')
result_df.reset_index(drop=True, inplace=True)


# 오답총합개수
fin_result = pd.DataFrame()
for uid in tqdm(result_df['U_INFO_ID'].unique()):
    new_temp = pd.DataFrame(columns = ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황'])
    new_temp.loc[0] = 0
    temp = result_df[result_df['U_INFO_ID'] == uid]

    for col in ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황']:
        new_temp[col] = round(temp[temp['CORRECT_YN'] == 0][col].iloc[0] / temp[temp['CORRECT_YN'] == 'total'][col].iloc[0], 2)
    new_temp['UID'] = uid
    fin_result = pd.concat([fin_result,new_temp])

fin_result = fin_result[['UID','교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명',
       '자전거, 친환경', '점검, 관리', '특별상황',]]

subj_result = pd.merge(subj_mst,fin_result, left_on = 'ID', right_on = 'UID' ,how='left')

print('End -- for --')

subj_result.to_csv('./subj_result.csv',index=False)

print('Finish!')


End  -- read file --


100%|███████████████████████████████████████████████████████████████████████████████| 6926/6926 [01:24<00:00, 81.53it/s]


End -- for --
Finish!


In [14]:
import warnings
import pandas as pd
import numpy as np
from multiprocessing import Pool
from functools import partial
from tqdm import tqdm

def process_individual_uid(uid, subj, subj_mst):
    temp = subj[subj['U_INFO_ID'] == uid]
    vs = sorted(temp['UNITCODE_NAME'].unique())

    if vs != ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황']:
        return None

    try:
        temp_pivot = temp.groupby(['U_INFO_ID', 'UNITCODE_NAME', 'CORRECT_YN'], as_index=False).agg({'U_NAME': 'count'}).pivot(index='CORRECT_YN', columns='UNITCODE_NAME', values='U_NAME').fillna(0).reset_index()

        if temp_pivot.shape[0] > 1:
            temp_pivot.columns.name = None
            temp_pivot.reset_index(drop=True, inplace=True)
            temp_pivot.loc[2, :] = temp_pivot.sum().to_list()
            temp_pivot.iloc[2, 0] = 'total'
            temp_pivot['U_INFO_ID'] = uid
            return temp_pivot

        elif temp_pivot.shape[0] == 1:
            temp_pivot.columns.name = None
            temp_pivot.reset_index(drop=True, inplace=True)
            temp_pivot['U_INFO_ID'] = uid
            temp_pivot.loc[1, :] = 0
            temp_pivot.sort_values('CORRECT_YN', ascending=True, inplace=True)
            temp_pivot.reset_index(drop=True, inplace=True)
            temp_pivot.loc[2, :] = temp_pivot.sum().to_list()
            temp_pivot.iloc[2, 0] = 'total'
            temp_pivot['U_INFO_ID'] = uid
            return temp_pivot

    except IndexError:
        print(uid)
        return None

def process_uids(uids, subj, subj_mst):
    result_df = pd.DataFrame()

    for uid in tqdm(uids):
        temp_result = process_individual_uid(uid, subj, subj_mst)
        if temp_result is not None:
            result_df = pd.concat([result_df, temp_result])

    return result_df

def process_individual_fin_result(uid, result_df):
    new_temp = pd.DataFrame(columns=['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황'])
    new_temp.loc[0] = 0
    temp = result_df[result_df['U_INFO_ID'] == uid]

    for col in ['교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명', '자전거, 친환경', '점검, 관리', '특별상황']:
        new_temp[col] = round(temp[temp['CORRECT_YN'] == 0][col].iloc[0] / temp[temp['CORRECT_YN'] == 'total'][col].iloc[0], 2)

    new_temp['UID'] = uid
    return new_temp

def process_fin_results(uids, result_df):
    fin_result = pd.DataFrame()

    for uid in tqdm(uids):
        temp_result = process_individual_fin_result(uid, result_df)
        fin_result = pd.concat([fin_result, temp_result])

    return fin_result

if __name__ == '__main__':
    warnings.filterwarnings('always')
    warnings.filterwarnings('ignore')

    subj = pd.read_csv('../../../011.데이터/채점데이터/_SELECT_ui_U_NAME_ui_U_JUMIN_NO_uec_U_INFO_ID_uec_Q_ID_qq_L_CD_q_20231101.csv', nrows=300000)
    subj_mst = pd.read_csv('../../../011.데이터/채점데이터/U_INFO_202311011052.csv')
    print('End  -- read file --')

    subj['BIRTH_NUM'] = subj['U_JUMIN_NO'].apply(lambda x: x[:6])
    subj['PER_IDX'] = subj['U_NAME'] + '_' + subj['BIRTH_NUM']

    mask1 = subj_mst['U_LEVEL'] == '보통'
    mask2 = subj_mst['U_PASSFAIL'] == 1
    subj_mst = subj_mst[mask1 & mask2]
    subj_mst = subj_mst[['ID', 'U_EXAM_OPEN', 'U_NAME', 'U_TYPE', 'U_LEVEL', ]]

    subj = subj[subj['U_INFO_ID'].isin(subj_mst['ID'].unique())]

    num_processes = 4  # 사용 가능한 CPU 코어 수에 따라 조정
    uids_chunks = np.array_split(subj['U_INFO_ID'].unique(), num_processes)

    process_uids_partial = partial(process_uids, subj=subj, subj_mst=subj_mst)

    with Pool(num_processes) as pool:
        results = pool.map(process_uids_partial, uids_chunks)

    result_df = pd.concat(results)
    result_df['U_INFO_ID'] = result_df['U_INFO_ID'].astype('int')
    result_df.reset_index(drop=True, inplace=True)

    uids_chunks_fin_result = np.array_split(result_df['U_INFO_ID'].unique(), num_processes)
    process_fin_results_partial = partial(process_fin_results, result_df=result_df)

    with Pool(num_processes) as pool:
        fin_results = pool.map(process_fin_results_partial, uids_chunks_fin_result)

    fin_result = pd.concat(fin_results)
    fin_result = fin_result[['UID', '교통사고', '도로', '동영상', '마음가짐', '면허취득', '법규준수', '안전운전', '안전표지', '인명',
                             '자전거, 친환경', '점검, 관리', '특별상황']]

    subj_result = pd.merge(subj_mst, fin_result, left_on='ID', right_on='UID', how='left')

    # 나머지 코드...
    subj_result.to_csv('./subj_result.csv', index=False)
    print('Finish!')


End  -- read file --


100%|███████████████████████████████████████████████████████████████████████████████| 1732/1732 [01:24<00:00, 20.38it/s]


Finish!


In [17]:
subj_result[~subj_result['UID'].isna()]

,ID,U_EXAM_OPEN,U_NAME,U_TYPE,U_LEVEL,UID,교통사고,도로,동영상,마음가짐,면허취득,법규준수,안전운전,안전표지,인명,"자전거, 친환경","점검, 관리",특별상황
0,35031,2022-01-03 09:00:00.000,한경아,2,보통,35031.0,0.0,0.11,0.0,0.00,0.0,1.00,0.43,0.2,0.0,0.33,0.00,0.5
1,35032,2022-01-03 09:00:00.000,김찬규,1,보통,35032.0,1.0,0.22,0.0,0.00,0.0,0.50,0.43,0.4,0.5,0.33,0.50,0.0
2,35033,2022-01-03 09:00:00.000,이호민,2,보통,35033.0,0.0,0.00,0.0,1.00,1.0,0.50,0.29,0.4,0.0,0.00,0.67,0.5
3,35034,2022-01-03 09:00:00.000,이진혁,2,보통,35034.0,0.0,0.11,0.0,0.00,1.0,0.50,0.43,0.0,0.0,0.00,0.00,1.0
4,35035,2022-01-03 09:00:00.000,박신유,2,보통,35035.0,0.0,0.22,1.0,1.00,1.0,0.50,0.14,0.2,0.0,0.33,1.00,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8205,43126,2022-01-24 09:00:00.000,이승우,2,보통,43126.0,0.0,0.00,0.0,0.67,1.0,0.75,0.43,0.4,0.5,0.00,0.50,1.0
8206,43128,2022-01-24 09:00:00.000,정선우,2,보통,43128.0,0.0,0.00,0.0,0.00,0.0,0.25,0.14,0.0,0.0,0.33,0.33,0.0
8207,43129,2022-01-24 09:00:00.000,박지원,2,보통,43129.0,0.0,0.00,0.0,0.00,1.0,0.25,0.57,0.2,0.5,0.00,0.67,1.0
8208,43130,2022-01-24 09:00:00.000,김윤중,1,보통,43130.0,0.0,0.11,0.0,0.33,1.0,0.50,0.43,0.0,0.0,0.33,0.50,0.0


# Function Test
- 시험 테이블과 시험 상세 테이블의 EXM_CODE가 형식이 다름
- 상세 테이블의 컬럼명 중 PNT가 점수와 관련된 컬럼임
- 외국인 발라내는 방법을 찾아야할듯..

In [2]:
func = pd.read_csv('../../../011.데이터/채점데이터/_SELECT_FROM_DISEXAM08_TB_RERRESTEXP_tr_inner_JOIN_DISEXAM08_TB__08_20231101.csv')
func = func[(func['LICN_CLS_CODE'].isin([12,32]))&(func['EXM_JUDG_CODE'] == 1)]

def re_birth(x):
    x = str(x)
    if len(x) == 6:
        return x
    elif len(x) == 5:
        return '0'+x
    elif len(x) == 4:
        return '00'+x
func['RESIDENT_DATE'] = func['RESIDENT_DATE'].apply(lambda x : re_birth(x))


In [6]:
tmp = func[['RESPOND_DATE', 'RESIDENT_DATE', 'RESIDENT_NAME', 'START_PNT', 'END_PNT', 'BELT_PNT','CROSS_PNT', 'CLIMB_PNT', 'TRAIN_PNT', 
      'SIGNAL_PNT', 'RPMOVER_PNT','SPEEDOVER_PNT', 'STARTING_PNT', 'ZCOUR_PNT', 'SCOUR_PNT', 'TCOUR_PNT','PCOUR_PNT', 'SUDDEN_PNT', 'GEAR_PNT', 'TIME_PNT',]]

In [9]:
tmp = tmp.groupby(['RESIDENT_DATE','RESIDENT_NAME'],as_index=False).agg({'RESPOND_DATE' :'count'})


In [19]:
func[func['RESIDENT_DATE'] == '001202']

,RESPOND_DATE,RES_GNUS_CODE,RES_CLS_CODE,RESPOND_TIME,EXM_NO,EXM_CODE,LICN_CON_CODE,RECEIVE_NO,RECV_EXM_CODE,RES_EXM_CODE,RECEIVE_DATE,RESIDENT_DATE,RESIDENT_NO,RESIDENT_NAME,HANDI_OPT,LICN_CLS_CODE,RES_START_TIME,POINT_OPT,EXM_JUDG_CODE,RESPOND_PNT,FAIL_CODE,RE_EXM_CNT,TRANS_OPT,ENTRY_DATE,ENTRY_TIME,ENTRY_NAME,UPDATE_DATE,UPDATE_TIME,FOREIGN_SUBJ,REM_GNUS_CODE,RESIDENT_NO$$,EXM_CHK_TIME,RESPOND_DATE.1,RES_GNUS_CODE.1,RES_CLS_CODE.1,RESPOND_TIME.1,EXM_NO.1,EXM_CODE.1,LICN_CON_CODE.1,FAIL_CODE.1,RESPOND_PNT.1,START_PNT,END_PNT,BELT_PNT,CROSS_PNT,CLIMB_PNT,TRAIN_PNT,SIGNAL_PNT,RPMOVER_PNT,SPEEDOVER_PNT,STARTING_PNT,ZCOUR_PNT,SCOUR_PNT,TCOUR_PNT,PCOUR_PNT,SUDDEN_PNT,GEAR_PNT,TIME_PNT,START_TIME,WRE_CONN_PNT,WRE_CONN_TIME_PNT,WRE_CONN_TIME,WRE_WIND_PNT,WRE_WIND_TIME_PNT,WRE_CURV_PNT,WRE_CURV_TIME_PNT,WRE_DIV_PNT,WRE_DIV_TIME_PNT,WRE_DIR_PNT,WRE_DIR_TIME_PNT,TRA_CONN_PNT,TRA_CONN_TIME_PNT,TRA_DIR_PNT,TRA_DIR_TIME_PNT,TRA_DIV_PNT,TRA_DIV_TIME_PNT,MOR_WIND_PNT,MOR_WIND_FOOT_PNT,MOR_CURV_PNT,MOR_CURV_FOOT_PNT,MOR_NARR_PNT,MOR_NARR_FOOT_PNT,MOR_CONT_PNT,MOR_CONT_FOOT_PNT,MOR_CONT_RUBB_PNT,END_TIME,UPDATE_DATE.1,UPDATE_TIME.1,HEADLIGHT_PNT,TURN_SIG_LMP_PNT,WIPER_PNT,GEAR_CHG_PNT,OBEY_CAR_ROAD_PNT,SUDDEN_STOP_PNT,LICN_CLS_CODE.1
6323,20220221,2,5,5,2506,6,6,8210039128,8,8,20220211,001202,NaN,김하늘,0,32,1600,2,1,95,0,0,A,20220221,60507,CRON_NEW,20220221.0,165841.0,NaN,NaN,2Addw2g8Afpq1cUkDZ1FHQAA==AAcA,NaN,20220221,2,5,5,2506,6,6,0.0,95,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,162635.0,20220221.0,165841.0,0.0,0.0,0.0,0.0,0.0,NaN,32
35439,20220926,2,5,3,2302,6,06,8220044228,8,8,20220924,001202,NaN,나예진,0,32,1300,2,1,88,0,0,A,20220926,60507,CRON_NEW,20220926.0,170716.0,NaN,NaN,NXx79pHacwfpY0eIsArdFAAA==AAcA,NaN,20220926,2,5,3,2302,6,06,0.0,88,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,12,NaN,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,133756.0,20220926.0,170716.0,0.0,0.0,0.0,0.0,0.0,NaN,32
52000,20230302,2,5,4,2414,6,06,8230010716,8,8,20230228,001202,NaN,김하늘,0,32,1430,2,1,80,0,0,A,20230302,60507,CRON_NEW,20230302.0,170906.0,NaN,NaN,2Addw2g8Afpq1cUkDZ1FHQAA==AAcA,NaN,20230302,2,5,4,2414,6,06,0.0,80,0,0,0,0,0,0,0,0,0,0,0,0,10,0,0,10,0,NaN,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,152630.0,20230302.0,170906.0,0.0,0.0,0.0,0.0,0.0,NaN,32


In [17]:
tmp[tmp['RESPOND_DATE'] > 1]

,RESIDENT_DATE,RESIDENT_NAME,RESPOND_DATE
135,001202,김하늘,2
466,010521,조시현,2
860,020131,양재민,2
1425,030112,JIN QUAN,2
1967,030804,이강우,2
2366,040106,조규상,2
2848,040630,정우진,2
3754,611029,최한동,2
4032,650208,조재철,2
4095,650911,김정호,2


In [4]:
[x for x in func.columns if 'PNT' in x and 'WRE' not in x and 'TRA' not in  x and 'MOR' not in x]

['RESPOND_PNT',
 'RESPOND_PNT.1',
 'START_PNT',
 'END_PNT',
 'BELT_PNT',
 'CROSS_PNT',
 'CLIMB_PNT',
 'SIGNAL_PNT',
 'RPMOVER_PNT',
 'SPEEDOVER_PNT',
 'STARTING_PNT',
 'ZCOUR_PNT',
 'SCOUR_PNT',
 'TCOUR_PNT',
 'PCOUR_PNT',
 'SUDDEN_PNT',
 'GEAR_PNT',
 'TIME_PNT',
 'HEADLIGHT_PNT',
 'TURN_SIG_LMP_PNT',
 'WIPER_PNT',
 'GEAR_CHG_PNT',
 'OBEY_CAR_ROAD_PNT',
 'SUDDEN_STOP_PNT']

In [32]:
func_det.head()

,RESPOND_DATE,RES_GNUS_CODE,RES_CLS_CODE,RESPOND_TIME,EXM_NO,EXM_CODE,LICN_CON_CODE,FAIL_CODE,RESPOND_PNT,START_PNT,END_PNT,BELT_PNT,CROSS_PNT,CLIMB_PNT,TRAIN_PNT,SIGNAL_PNT,RPMOVER_PNT,SPEEDOVER_PNT,STARTING_PNT,ZCOUR_PNT,SCOUR_PNT,TCOUR_PNT,PCOUR_PNT,SUDDEN_PNT,GEAR_PNT,TIME_PNT,START_TIME,WRE_CONN_PNT,WRE_CONN_TIME_PNT,WRE_CONN_TIME,WRE_WIND_PNT,WRE_WIND_TIME_PNT,WRE_CURV_PNT,WRE_CURV_TIME_PNT,WRE_DIV_PNT,WRE_DIV_TIME_PNT,WRE_DIR_PNT,WRE_DIR_TIME_PNT,TRA_CONN_PNT,TRA_CONN_TIME_PNT,TRA_DIR_PNT,TRA_DIR_TIME_PNT,TRA_DIV_PNT,TRA_DIV_TIME_PNT,MOR_WIND_PNT,MOR_WIND_FOOT_PNT,MOR_CURV_PNT,MOR_CURV_FOOT_PNT,MOR_NARR_PNT,MOR_NARR_FOOT_PNT,MOR_CONT_PNT,MOR_CONT_FOOT_PNT,MOR_CONT_RUBB_PNT,END_TIME,UPDATE_DATE,UPDATE_TIME,HEADLIGHT_PNT,TURN_SIG_LMP_PNT,WIPER_PNT,GEAR_CHG_PNT,OBEY_CAR_ROAD_PNT,SUDDEN_STOP_PNT,LICN_CLS_CODE
0,20220112,2,5,2,1201,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32
1,20220113,2,5,2,2210,6,6,0.0,90,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10,0,0,NaN,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,110644.0,20220113.0,165412.0,0.0,0.0,0.0,0.0,0.0,NaN,32
2,20220113,2,5,3,2306,6,6,99.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,132006.0,20220113.0,165413.0,5.0,0.0,0.0,0.0,0.0,NaN,32
3,20220113,2,5,5,2521,6,6,0.0,77,0,0,0,0,0,0,0,0,0,0,0,0,20,0,0,0,3,NaN,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,163515.0,20220113.0,165414.0,0.0,0.0,0.0,0.0,0.0,NaN,32
4,20220113,2,5,4,2416,6,6,0.0,100,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,150914.0,20220113.0,165413.0,0.0,0.0,0.0,0.0,0.0,NaN,32


# Road Drive Test

In [20]:
road = pd.read_csv('../../../011.데이터/채점데이터/TB_RERROADEXP_08_20231101.csv')


In [23]:
[x for x in road.columns if 'PNT' in x]

['RESPOND_PNT',
 'BEF_DECT_PNT',
 'DRV_DECT_PNT',
 'STR_DECT_PNT',
 'SPD_DECT_PNT',
 'BRAKE_DECT_PNT',
 'STER_DECT_PNT',
 'BODY_DECT_PNT',
 'NEVI_DECT_PNT',
 'CHG_DECT_PNT',
 'TURN_DECT_PNT',
 'PARK_DECT_PNT',
 'ETC_DECT_PNT']

In [21]:
road.head()

,EXM_LICEN_NO,RECEIVE_DATE,RECEIVE_TIME,LICN_PRT_DATE,RESIDENT_DATE,RESIDENT_NO,RESIDENT_NAME,RES_PHONE_NO,PROC_EXM_CODE,RES_EXM_CODE,EXM_CODE,RES_CLS_CODE,LICN_CON_CODE,RESPOND_DATE,RESPOND_TIME,RES_START_TIME,EXM_NO,RECV_KIND_CODE,WEB_RECV_OPT,COURSE,OBSRV_EXM_NO,OBSRV_RESIDENT_DATE,OBSRV_RESIDENT_NO,OBSRV_RESIDENT_NAME,OBSRV_PHONE_NO,RESPOND_PNT,EXM_JUDG_CODE,WRONG_CNT,MANUAL_OPT,CAR_STR_TIME,CAR_END_TIME,PARK_STR_TIME,PARK_END_TIME,BEF_DECT_PNT,DRV_DECT_PNT,STR_DECT_PNT,SPD_DECT_PNT,BRAKE_DECT_PNT,STER_DECT_PNT,BODY_DECT_PNT,NEVI_DECT_PNT,CHG_DECT_PNT,TURN_DECT_PNT,PARK_DECT_PNT,ETC_DECT_PNT,FAIL_CODE,CAR_NO,RSLT_ENT_DATE,RSLT_ENT_ID,RSLT_ENT_NAME,ROAD_EXM_OPT,PARK_EXM_OPT,LIFEBELT_OPT,HANDICAP_OPT,TABLET_IP_ADDR,POINT_OPT,ENTRY_PLACE,ENTRY_DATE,ENTRY_TIME,ENTRY_NAME,UPDATE_DATE,UPDATE_TIME,DOWNLOAD_YN,AUTO_YN,PARK_ENTER_TIME,SPEED_CHECK40,SPEED_CHECK50,SPEED_CHECK60,SPEED_CHECK70,DISTANCE
0,58210346451,20211221,121649,20211123,20803,3042117,최우석,1.031422e+09,8,8,06,32,06,20220103,2,1000,4108,2,NaN,A,4102.0,920221.0,1000000.0,강대인,1.080089e+09,86.0,1,0.0,N,104138.0,105923.0,0.0,0.0,0,0,0,0,0,0,0,0,7,7,0,0,0.0,2-9(9164),20220103.0,1101348.0,정영제,Y,N,N,N,D0:7F:A0:CF:E7:47,2,8,20220103,605,CRON,20220103.0,112809.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
1,58210383204,20211231,100408,20211221,961016,1041511,남상백,1.088931e+09,8,8,06,32,06,20220103,4,1430,4102,0,NaN,A,4108.0,991231.0,1000000.0,장혁,1.049133e+09,100.0,1,0.0,N,153037.0,154902.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,2-12(9174),20220103.0,1112106.0,최유정,Y,N,N,N,D0:7F:A0:CF:E7:99,2,8,20220103,605,CRON,20220103.0,155114.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
2,58210235674,20211231,153606,20210730,11007,3051417,임승호,1.032278e+09,8,8,00,12,00,20220103,4,1430,4006,2,NaN,A,4001.0,890821.0,1000000.0,정청,1.020816e+09,81.0,1,0.0,N,145929.0,152223.0,0.0,0.0,0,0,0,5,0,7,0,0,7,0,0,0,0.0,1-1(9579),20220103.0,1101448.0,김종모,Y,N,N,N,D0:7F:A0:CF:E7:5F,2,8,20220103,605,CRON,20220103.0,152114.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
3,58210397548,20211231,143129,20211231,870422,1201020,오민학,1.084606e+09,8,8,00,12,00,20220103,4,1430,4005,1,NaN,A,4002.0,731003.0,1000000.0,구백서,1.062531e+09,83.0,1,0.0,N,150902.0,152848.0,0.0,0.0,0,0,0,0,5,0,0,5,7,0,0,0,0.0,1-5(5234),20220103.0,1207093.0,박성민,Y,N,N,N,D0:7F:A0:CF:E7:B1,2,8,20220103,605,CRON,20220103.0,153015.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
4,58210314617,20211231,120015,20211021,750507,2090625,양유정,1.033574e+09,0,8,06,32,06,20220103,3,1300,4117,0,NaN,B,4111.0,900309.0,5000000.0,ZHANGLONGSHAN,NaN,0.0,2,0.0,N,132946.0,133116.0,0.0,0.0,0,0,0,0,0,0,0,0,0,7,0,0,9.0,2-5(1832),20220103.0,1101238.0,신유선,Y,N,N,N,D0:7F:A0:CF:E7:B9,2,0,20220103,605,CRON,20220103.0,133444.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN


In [188]:
road_pnt.head()

,EXM_LICEN_NO,EXAMDATE,CLASS,LICENSE,EXAMTYPE,APPNO,TESTNO,COURSEFLAG,POINTCODE,POINTTIME,CANCELFLAG,REFFLAG,REFVALUE,MANUALFLAG,CANCELREASON,LATITUDE,LONGITUDE
0,59210385191,20220105,4,2A,A,10,1,1,32,2022-01-05 14:47:04.000,N,0,0,N,NaN,37.578970,126.880617
1,59210385191,20220105,4,2A,A,10,1,1,101,2022-01-05 14:48:42.000,N,0,0,N,NaN,37.578307,126.881932
2,59210385191,20220105,4,2A,A,10,1,1,61,2022-01-05 14:49:00.000,N,0,0,Y,NaN,37.578260,126.881887
3,59210385191,20220105,4,2A,A,10,1,1,41,2022-01-05 14:51:16.000,N,0,0,N,NaN,37.578707,126.878853
4,59210385191,20220105,4,2A,A,10,1,1,23,2022-01-05 14:55:45.000,N,0,0,Y,NaN,37.574323,126.886207


In [ ]:
road_pnt

In [190]:
road.shape

(20651, 70)

In [194]:
road

,EXM_LICEN_NO,RECEIVE_DATE,RECEIVE_TIME,LICN_PRT_DATE,RESIDENT_DATE,RESIDENT_NO,RESIDENT_NAME,RES_PHONE_NO,PROC_EXM_CODE,RES_EXM_CODE,EXM_CODE,RES_CLS_CODE,LICN_CON_CODE,RESPOND_DATE,RESPOND_TIME,RES_START_TIME,EXM_NO,RECV_KIND_CODE,WEB_RECV_OPT,COURSE,OBSRV_EXM_NO,OBSRV_RESIDENT_DATE,OBSRV_RESIDENT_NO,OBSRV_RESIDENT_NAME,OBSRV_PHONE_NO,RESPOND_PNT,EXM_JUDG_CODE,WRONG_CNT,MANUAL_OPT,CAR_STR_TIME,CAR_END_TIME,PARK_STR_TIME,PARK_END_TIME,BEF_DECT_PNT,DRV_DECT_PNT,STR_DECT_PNT,SPD_DECT_PNT,BRAKE_DECT_PNT,STER_DECT_PNT,BODY_DECT_PNT,NEVI_DECT_PNT,CHG_DECT_PNT,TURN_DECT_PNT,PARK_DECT_PNT,ETC_DECT_PNT,FAIL_CODE,CAR_NO,RSLT_ENT_DATE,RSLT_ENT_ID,RSLT_ENT_NAME,ROAD_EXM_OPT,PARK_EXM_OPT,LIFEBELT_OPT,HANDICAP_OPT,TABLET_IP_ADDR,POINT_OPT,ENTRY_PLACE,ENTRY_DATE,ENTRY_TIME,ENTRY_NAME,UPDATE_DATE,UPDATE_TIME,DOWNLOAD_YN,AUTO_YN,PARK_ENTER_TIME,SPEED_CHECK40,SPEED_CHECK50,SPEED_CHECK60,SPEED_CHECK70,DISTANCE
0,58210387199,20220105,100132,20211224,20803,3068011,정인하,1.023096e+09,8,8,06,32,06,20220105,3,1300,4120,0,NaN,A,4105.0,990504.0,1000000.0,정상일,1.086533e+09,0.0,2,0.0,N,133100.0,134637.0,0.0,0.0,0,0,0,0,0,0,0,0,27,7,0,0,13.0,2-4(1484),20220105.0,1101238.0,신유선,Y,N,N,N,D0:7F:A0:CF:E7:B9,2,8,20220105,1002,CRON,20220105.0,143453.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
1,58220003763,20220105,100152,20220105,880711,1285516,박기형,1.058821e+09,8,8,06,32,06,20220105,3,1300,4121,1,NaN,B,4104.0,970423.0,1000000.0,최주호,1.040297e+09,54.0,2,0.0,N,140330.0,142157.0,0.0,0.0,0,5,0,0,0,7,0,0,20,14,0,0,0.0,2-11(9168),20220105.0,7001431.0,강혜진,Y,N,N,N,D0:B1:28:05:01:C9,2,8,20220105,1002,CRON,20220105.0,143453.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
2,58220116733,20220401,93856,20220401,930405,1321013,김재경,1.022955e+09,8,8,06,32,06,20220401,2,1000,4116,1,NaN,A,4104.0,20110.0,3000000.0,박지환,1.055570e+09,93.0,1,0.0,N,103455.0,105235.0,0.0,0.0,0,0,0,0,0,7,0,0,0,0,0,0,0.0,2-12(9174),20220401.0,1101241.0,임선업,Y,N,N,N,D0:B1:28:05:03:4D,2,8,20220401,941,CRON,20220401.0,111459.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
3,58210351091,20220414,95248,20211126,960905,1047711,윤준식,1.090567e+09,8,8,06,32,06,20220414,2,1000,4116,0,NaN,C,4103.0,850711.0,1000000.0,장성후,1.039493e+09,0.0,2,0.0,N,104506.0,104703.0,0.0,0.0,0,0,0,0,0,0,0,5,0,0,0,0,9.0,2-12(9174),20220414.0,1207093.0,박성민,Y,N,N,N,D0:7F:A0:CF:E7:B1,2,8,20220414,957,CRON,20220414.0,112232.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
4,58220128057,20220414,95416,20220414,891125,1030611,황동헌,1.084603e+09,8,8,06,32,06,20220414,3,1300,4110,1,NaN,D,4112.0,811117.0,2000000.0,전민서,1.049034e+09,75.0,1,0.0,N,134146.0,140031.0,0.0,0.0,0,20,5,0,0,0,0,0,0,0,0,0,0.0,2-11(9168),20220414.0,7000482.0,장일주,Y,N,N,N,D0:B1:28:05:05:A3,2,8,20220414,957,CRON,20220414.0,135937.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20646,58220279662,20221121,143241,20220920,760602,1000228,이상용,1.022473e+09,8,8,00,12,00,20221121,5,1600,4001,2,NaN,C,4003.0,830717.0,1000000.0,김진우,1.033864e+09,81.0,1,0.0,N,162220.0,164355.0,0.0,0.0,0,0,0,0,5,0,0,0,7,7,0,0,0.0,1-8(9315),20221121.0,1101241.0,임선업,Y,N,N,N,D0:B1:28:05:03:4D,2,8,20221121,1436,CRON,20221121.0,171927.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
20647,58220329088,20221123,94058,20221123,670619,1657513,이이범,NaN,8,8,00,12,00,20221123,2,1000,4001,1,NaN,B,4104.0,961126.0,2000000.0,강혜림,1.099145e+09,0.0,2,0.0,N,102215.0,102515.0,0.0,0.0,0,0,0,0,5,0,0,0,7,17,0,0,5.0,1-8(9315),20221123.0,1101205.0,이정화,Y,N,N,N,D0:B1:28:05:03:19,2,8,20221123,940,CRON,20221123.0,170456.0,Y,Y,0.0,0.0,0.0,0.0,0.0,NaN
20648,58220328290,20221122,112753,20221122,790213,1030124,오근찬,1.044776e+09,8,8,06,32,06,20221122,3,1300,4106,1,NaN,A,4109.0,510510.0,5000000.0,KIMYOUNGKU,NaN,83.0,1,0.0,N,133047.0,134916.0,0.0,0.0,0,5,0,0,0,0,0,5,7,0,0,0,0.0,2-8(4153),20221122.0,1207093.0,박성민,Y,N,N,N,D0:7F:A0:CF:E7:B1,2,8,20221122,1131,CRON,20221122.0,143502.0